In [1]:
import sys
from pathlib import Path

# This notebook lives in notebooks/, so the project root is one level up
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: d:\My_projects\data-engineer\waste-food-project


In [2]:
from sqlalchemy import select, text, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

# Shared engine (reads DATABASE_URL from the project's .env file)
from db.config.connection import engine


# Minimal model for the `country` table (public schema, no schema prefix).
# CRUD examples below only need this table.
class Base(DeclarativeBase):
    pass


class Country(Base):
    __tablename__ = "country"

    m49_code: Mapped[str] = mapped_column(String(3), primary_key=True)
    country_name: Mapped[str] = mapped_column(String(150), unique=True, nullable=False)


print("SQLAlchemy imports completed successfully!")


SQLAlchemy imports completed successfully!


## 1. Test Database Connection

Verify that SQLAlchemy is successfully connected to the `food_waste_db` PostgreSQL database.

In [3]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database();")
    )

    print("Connected successfully!")
    print("Database:", result.scalar())

Connected successfully!
Database: food_waste_db


## 2. CREATE – Insert Records

Insert a new test country into the database using SQLAlchemy ORM.

We will use `session.add()` to add the record and `session.commit()` to save it permanently.

In [4]:
with Session(engine) as session:

    new_country = Country(
        m49_code="995",
        country_name="Notebook Test Country"
    )

    session.add(new_country)
    session.commit()

    print("Country inserted successfully!")
    print(f"Code: {new_country.m49_code}")
    print(f"Name: {new_country.country_name}")

Country inserted successfully!
Code: 995
Name: Notebook Test Country


### 2.1 Verify Inserted Record

Retrieve the inserted country using its primary key to confirm that it exists in the database.

In [5]:
with Session(engine) as session:

    country = session.get(Country, "995")

    if country:
        print("Country found!")
        print(f"Code: {country.m49_code}")
        print(f"Name: {country.country_name}")
    else:
        print("Country not found!")

Country found!
Code: 995
Name: Notebook Test Country


### 2.2 READ – Query Records

Read data with `select()`: count the rows, list a few countries, and filter by name.

In [6]:
from sqlalchemy import func

with Session(engine) as session:

    total = session.scalar(select(func.count()).select_from(Country))
    print("Total countries:", total)

    print("\nFirst 5 countries:")
    for country in session.scalars(select(Country).order_by(Country.country_name).limit(5)):
        print(f"- {country.m49_code}: {country.country_name}")

    print("\nFilter by name (contains 'Egypt'):")
    for country in session.scalars(
        select(Country).where(Country.country_name.ilike("%Egypt%"))
    ):
        print(f"- {country.m49_code}: {country.country_name}")


Total countries: 143

First 5 countries:
- 4: Afghanistan
- 8: Albania
- 12: Algeria
- 24: Angola
- 32: Argentina

Filter by name (contains 'Egypt'):
- 818: Egypt


## 3. UPDATE – Modify Records

Update an existing country using SQLAlchemy ORM.

We retrieve the record, modify its attribute, and call `session.commit()` to save the changes.

In [7]:
with Session(engine) as session:

    country = session.get(Country, "995")

    if country:
        country.country_name = "Updated Notebook Country"

        session.commit()

        print("Country updated successfully!")
        print(f"New Name: {country.country_name}")
    else:
        print("Country not found!")

Country updated successfully!
New Name: Updated Notebook Country


## 3.1 Verify Updated Record

Retrieve the country again to confirm that the updated value was saved.

In [8]:
with Session(engine) as session:

    country = session.get(Country, "995")

    if country:
        print(f"Code: {country.m49_code}")
        print(f"Updated Name: {country.country_name}")

Code: 995
Updated Name: Updated Notebook Country


## 4. DELETE – Remove Records

Delete the test country from the database using `session.delete()`.

Call `session.commit()` to permanently save the deletion.

In [9]:
with Session(engine) as session:

    country = session.get(Country, "995")

    if country:
        session.delete(country)
        session.commit()

        print("Country deleted successfully!")
    else:
        print("Country not found!")

Country deleted successfully!


### 4.1 Verify Deleted Record

Check whether the test country still exists in the database.

In [10]:
with Session(engine) as session:

    country = session.get(Country, "995")

    if country is None:
        print("Verified: Country was deleted.")
    else:
        print("Country still exists!")

Verified: Country was deleted.


## 5. Transactions – COMMIT & ROLLBACK

A transaction is a group of database operations treated as one unit.

- `commit()`: Permanently saves changes.
- `rollback()`: Cancels uncommitted changes.

In [11]:
with Session(engine) as session:

    new_country = Country(
        m49_code="994",
        country_name="Commit Test Country"
    )

    session.add(new_country)
    session.commit()

    print("COMMIT: Record saved successfully!")

COMMIT: Record saved successfully!


### 5.1 ROLLBACK – Cancel Changes

Insert a temporary record, then use `rollback()` before committing.

The inserted record should not be saved.

In [12]:
with Session(engine) as session:

    new_country = Country(
        m49_code="993",
        country_name="Rollback Test Country"
    )

    session.add(new_country)
    session.rollback()

    print("ROLLBACK: Changes cancelled!")

ROLLBACK: Changes cancelled!


### 5.2 Verify Transaction Results

Check that the committed record exists and the rolled-back record does not exist.

In [13]:
with Session(engine) as session:

    committed = session.get(Country, "994")
    rolled_back = session.get(Country, "993")

    print("994 exists:", committed is not None)
    print("993 exists:", rolled_back is not None)

994 exists: True
993 exists: False


## 6. Error Handling – Handling Database Errors

SQLAlchemy may raise errors when database constraints are violated.

We will test a duplicate Primary Key and handle the error using:

- `try / except`
- `IntegrityError`
- `rollback()`

In [14]:
from sqlalchemy.exc import IntegrityError

with Session(engine) as session:

    try:
        duplicate_country = Country(
            m49_code="994",
            country_name="Another Test Country"
        )

        session.add(duplicate_country)
        session.commit()

    except IntegrityError as error:
        session.rollback()

        print("Integrity Error: Duplicate Primary Key!")
        print("Transaction rolled back successfully.")

Integrity Error: Duplicate Primary Key!
Transaction rolled back successfully.


## 7. Cleanup – Remove Test Data

Delete the temporary test country used in the COMMIT and Error Handling examples.

This keeps the original project database clean.

In [15]:
with Session(engine) as session:

    country = session.get(Country, "994")

    if country:
        session.delete(country)
        session.commit()
        print("Test data cleaned successfully!")
    else:
        print("No test data found.")

Test data cleaned successfully!


## 8. Summary – SQLAlchemy CRUD & Transactions

In this notebook, we practiced:

1. Database Connection
2. CREATE – INSERT records
3. READ – SELECT records
4. UPDATE – Modify records
5. DELETE – Remove records
6. COMMIT & ROLLBACK – Manage transactions
7. Error Handling – Handle database constraint violations

All operations were tested successfully using PostgreSQL and SQLAlchemy ORM.